<div dir="rtl">

# نوت‌بوک ۱: ممیزی ساختار داده

**فاز ۱، تسک ۴**

این نوت‌بوک به یک سؤال جواب می‌دهد: **داده‌ی ما دقیقاً چه شکلی است؟**

- هر فایل چند ردیف و چند ستون دارد و هر ستون از چه نوعی است؟
- شناسه‌ها (محصول، نسخه‌ی محصول، کاربر) چطور کار می‌کنند؟
- دو فایل نقدها و اطلاعات محصول چطور به هم وصل‌اند؟
- متغیر زمان چه بازه‌ای را پوشش می‌دهد؟
- زیرمجموعه‌ی کنترل از راه دور از چه زیرشاخه‌هایی ساخته شده است؟

ورودی: خروجی `src/build_subset.py` در `data/processed/`.
بررسی کیفیت (مقادیر گمشده، تکراری، پرت) موضوع نوت‌بوک ۲ است.

</div>

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load(open(ROOT / "configs" / "config.yaml", encoding="utf-8"))
PROC = ROOT / cfg["data"]["processed_dir"]
FIG = ROOT / "figures"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

reviews = pd.read_parquet(PROC / "reviews_rc.parquet")
meta = pd.read_parquet(PROC / "meta_rc.parquet")
audit = json.load(open(ROOT / "reports" / "phase1" / "raw_schema_audit.json"))

print(f"reviews: {reviews.shape[0]:,} rows x {reviews.shape[1]} columns")
print(f"meta:    {meta.shape[0]:,} rows x {meta.shape[1]} columns")

<div dir="rtl">

## ۱. ستون‌ها و نوع داده

برای هر ستون: نوع ذخیره‌شده، درصد ردیف‌هایی که مقدار دارند، و یک نمونه مقدار.
ستون‌هایی که با «JSON» مشخص شده‌اند در فایل خام لیست یا دیکشنری بوده‌اند و به‌صورت متن ذخیره شده‌اند تا ساختار جدول یکدست بماند.

</div>

In [ ]:
def column_table(df):
    rows = []
    for c in df.columns:
        s = df[c]
        non_null = s.notna().mean() * 100
        example = s.dropna().iloc[0] if s.notna().any() else None
        if isinstance(example, str) and len(example) > 60:
            example = example[:60] + "..."
        rows.append({"column": c, "dtype": str(s.dtype),
                     "non_null_%": round(non_null, 1), "example": example})
    return pd.DataFrame(rows)

column_table(reviews)

In [ ]:
column_table(meta)

<div dir="rtl">

## ۲. ناسازگاری نوع در فایل خام

اسکریپت ساخت زیرمجموعه برای **هر فیلد در هر خط** از ۱۷ میلیون خط خام شمرده است که چه نوع مقداری داشته.
فیلدهایی که بیش از یک نوع داشته‌اند در جدول زیر آمده‌اند.

- `missing` یعنی فیلد در خط بوده ولی مقدارش خالی (null) بوده است.
- `absent` یعنی فیلد اصلاً در آن خط وجود نداشته است.

این شمارش روی **کل دسته‌ی اسباب‌بازی** است، نه فقط زیرمجموعه.

</div>

In [ ]:
rows = []
for part in ("reviews", "meta"):
    for field, types in audit["field_types"][part].items():
        total = sum(types.values())
        for t, n in types.items():
            rows.append({"file": part, "field": field, "type": t,
                         "count": n, "share_%": round(100 * n / total, 2)})
types_df = pd.DataFrame(rows)

n_types = types_df.groupby(["file", "field"])["type"].nunique()
mixed = n_types[n_types > 1].index
types_df.set_index(["file", "field"]).loc[mixed].reset_index()

<div dir="rtl">

## ۳. شناسه‌ها

سه شناسه داریم:

| شناسه | معنی |
|---|---|
| `parent_asin` | محصول (همه‌ی رنگ‌ها و اندازه‌ها با هم) و **واحد محصول ما** |
| `asin` | یک نسخه‌ی مشخص (مثلاً رنگ قرمز) |
| `user_id` | نقدنویس؛ فقط شناسه‌ی تحلیلی است و برای شناسایی افراد استفاده نمی‌شود |

</div>

In [ ]:
ids = pd.Series({
    "products in meta (rows)": len(meta),
    "unique parent_asin in meta": meta["parent_asin"].nunique(),
    "unique parent_asin in reviews": reviews["parent_asin"].nunique(),
    "unique asin (variants) in reviews": reviews["asin"].nunique(),
    "unique user_id in reviews": reviews["user_id"].nunique(),
})
ids.map("{:,}".format)

<div dir="rtl">

### چند نسخه برای هر محصول؟ چند نقد برای هر کاربر؟

</div>

In [ ]:
variants = reviews.groupby("parent_asin")["asin"].nunique()
per_user = reviews.groupby("user_id").size()

summary = pd.DataFrame({
    "variants per product": variants.describe(percentiles=[.5, .9, .99]),
    "reviews per user": per_user.describe(percentiles=[.5, .9, .99]),
}).round(2)
summary

<div dir="rtl">

## ۴. ارتباط دو فایل

رابطه‌ی دو فایل **چند به یک** است: هر نقد دقیقاً به یک محصول تعلق دارد و هر محصول می‌تواند صفر یا چند نقد داشته باشد. کلید اتصال `parent_asin` است.

</div>

In [ ]:
meta_ids = set(meta["parent_asin"])
review_ids = set(reviews["parent_asin"])

link = pd.Series({
    "reviews whose product is in meta": reviews["parent_asin"].isin(meta_ids).sum(),
    "reviews whose product is NOT in meta": (~reviews["parent_asin"].isin(meta_ids)).sum(),
    "products with at least one review": len(meta_ids & review_ids),
    "products with no review": len(meta_ids - review_ids),
})
link.map("{:,}".format)

In [ ]:
per_product = reviews.groupby("parent_asin").size()
per_product.describe(percentiles=[.25, .5, .75, .9, .99]).round(1)

<div dir="rtl">

## ۵. متغیر زمان

`timestamp` زمان ثبت نقد به میلی‌ثانیه از ۱ ژانویه‌ی ۱۹۷۰ (Unix time) است. آن را به تاریخ تبدیل می‌کنیم.
بررسی کامل الگوهای زمانی در نوت‌بوک تحلیل اکتشافی می‌آید؛ اینجا فقط بازه و پوشش را می‌بینیم.

</div>

In [ ]:
reviews["date"] = pd.to_datetime(reviews["timestamp"], unit="ms", utc=True)
print("first review:", reviews["date"].min())
print("last review: ", reviews["date"].max())
print("missing timestamps:", reviews["timestamp"].isna().sum())

reviews["date"].dt.year.value_counts().sort_index().to_frame("reviews")

<div dir="rtl">

## ۶. زیرشاخه‌های زیرمجموعه

مسیر دسته‌بندی آمازون (`categories`) را باز می‌کنیم و سطح سوم آن را می‌شماریم.
این نشان می‌دهد زیرمجموعه چقدر «وسیله‌ی کامل» (ماشین، پهپاد، هلی‌کوپتر) و چقدر «قطعه و لوازم جانبی» است.
تصمیم تسک ۴: قطعات نگه داشته می‌شوند و در فاز ۴ به‌عنوان زیرگروه مقایسه می‌شوند.

</div>

In [ ]:
def level(cats_json, k):
    cats = json.loads(cats_json) if cats_json else []
    return cats[k] if len(cats) > k else "(none)"

meta["cat_level2"] = meta["categories"].map(lambda c: level(c, 1))
meta["cat_level3"] = meta["categories"].map(lambda c: level(c, 2))

n_reviews = reviews.groupby("parent_asin").size()
meta["n_reviews"] = meta["parent_asin"].map(n_reviews).fillna(0).astype(int)

sub = (meta.groupby("cat_level3")
           .agg(products=("parent_asin", "size"), reviews=("n_reviews", "sum"))
           .sort_values("reviews", ascending=False))
sub["reviews_%"] = (100 * sub["reviews"] / sub["reviews"].sum()).round(1)
sub

<div dir="rtl">

نمونه‌ای از عنوان محصولات در سه زیرشاخه‌ی پرنقد:

</div>

In [ ]:
for name in sub.index[:3]:
    print(f"--- {name} ---")
    titles = meta.loc[meta["cat_level3"] == name, "title"].dropna()
    for t in titles.sample(min(5, len(titles)), random_state=cfg["seed"]):
        print("  ", t[:110])

<div dir="rtl">

## ۷. نمودار جریان داده

از فایل‌های خام تا جدول‌هایی که در بقیه‌ی پروژه استفاده می‌شوند.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.set_xlim(0, 11)
ax.set_ylim(0, 4.2)
ax.axis("off")

def box(x, y, w, h, title, sub, color):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="#444", lw=1))
    ax.text(x + w / 2, y + h * 0.64, title, ha="center", va="center", fontsize=10, weight="bold")
    ax.text(x + w / 2, y + h * 0.3, sub, ha="center", va="center", fontsize=8.5)

def arrow(x1, y1, x2, y2, label=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color="#444", lw=1.2))
    if label:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.15, label, ha="center", fontsize=8, color="#333")

raw, mid, out = "#e8eef7", "#fdf3e1", "#e6f4ea"
box(0.2, 2.5, 2.8, 1.2, "meta_Toys_and_Games", "890,874 products (0.66 GB)", raw)
box(0.2, 0.5, 2.8, 1.2, "Toys_and_Games", "16,260,406 reviews (1.95 GB)", raw)
box(4.1, 1.5, 2.8, 1.2, "build_subset.py", "keep RC category tree\n+ raw type audit", mid)
box(8.0, 2.5, 2.8, 1.2, "meta_rc.parquet", f"{len(meta):,} products", out)
box(8.0, 0.5, 2.8, 1.2, "reviews_rc.parquet", f"{len(reviews):,} reviews", out)

arrow(3.0, 3.1, 4.1, 2.3)
arrow(3.0, 1.1, 4.1, 1.9)
arrow(6.9, 2.3, 8.0, 3.1)
arrow(6.9, 1.9, 8.0, 1.1)
ax.annotate("", xy=(9.4, 1.7), xytext=(9.4, 2.5),
            arrowprops=dict(arrowstyle="<-", color="#2a7", lw=1.2))
ax.text(9.55, 2.1, "many-to-one\njoin on parent_asin", fontsize=8, color="#2a7", va="center")

plt.tight_layout()
FIG.mkdir(exist_ok=True)
plt.savefig(FIG / "fig01_data_flow.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۸. جمع‌بندی

_این بخش را بعد از اجرای نوت‌بوک و دیدن خروجی‌ها با هم می‌نویسیم._

</div>